# «Фортуна»: Qwen3.5-4B LoRA для планировщика

Первый эксперимент: модель выбирает существующие операции приложения и выдаёт JSON. Задания синтетические; файловые операции здесь **не выполняются**. Это не обучение управления мышью/окнами и не измерение выполненных задач на ПК.

1. В проекте запустите `.venv\Scripts\python.exe scripts/prepare_fortuna_colab.py`.
2. В Colab включите GPU. Установите зависимости первой ячейкой.
3. Загрузите полученный `.training-output/fortuna-colab.zip`.
4. Сначала выполните короткий запуск с `SMOKE_TEST=True`. Для полного запуска начните новый сеанс и смените на `False`.

[Инструкция и требования Unsloth](https://unsloth.ai/docs/models/qwen3.5/fine-tune). QLoRA выключено. Ориентир BF16 LoRA 4B — 10 ГБ VRAM, но контекст и окружение влияют на расход. BF16 выбирается только при поддержке GPU. GPU-обучение и экспорт этого комплекта ещё требуют проверки в вашем сеансе Colab.


In [ ]:
# Устанавливается только в Colab, не в окружение приложения.
# Если Colab просит перезапуск: перезапустите сеанс и продолжите следующей ячейкой.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade",
    "unsloth==2026.10.5", "unsloth_zoo==2026.10.5",
    "transformers==5.2.0", "trl==0.22.2", "jsonschema", "pillow"])


In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Unsloth импортируется раньше transformers/trl.
from unsloth import FastLanguageModel
import torch, json, hashlib, time, zipfile, shutil, gc
from pathlib import Path
from importlib.metadata import version
from google.colab import files

SMOKE_TEST = True
MAX_SEQ_LENGTH = 4096
EXPORT_GGUF = False                 # Включите после полного сравнения и просмотра ответов.
RUN_DIR = Path("/content/fortuna-output")
RESUME_FROM_CHECKPOINT = None        # Путь к checkpoint того же запуска/набора данных.

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в меню «Среда выполнения → Сменить среду выполнения».")
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), "VRAM:", round(total_bytes / 2**30, 1), "GiB",
      "свободно:", round(free_bytes / 2**30, 1), "GiB")
if free_bytes < 10 * 2**30:
    raise RuntimeError("Для этого LoRA-пилота нужно больше свободной VRAM. Выберите более вместительную GPU.")
USE_BF16 = torch.cuda.is_bf16_supported()
print("Тип обучения:", "BF16" if USE_BF16 else "FP16", "| короткий запуск:", SMOKE_TEST)
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise RuntimeError("Каталог результатов уже занят. Скачайте его и выберите новый RUN_DIR или новый сеанс.")
RUN_DIR.mkdir(parents=True, exist_ok=True)


## Загрузка подготовленного архива

Выберите только `fortuna-colab.zip`. Проверяются состав, хеши, схемы, зависимости и разделение train/validation/test. Код проверки внутри архива — часть вашего локально подготовленного комплекта.


In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Выберите один файл fortuna-colab.zip.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.endswith(".zip"):
    raise ValueError("Нужен архив, созданный prepare_fortuna_colab.py.")
BUNDLE = Path("/content/fortuna-bundle")
BUNDLE.mkdir(exist_ok=True)
allowed = {"contract.json", "manifest.json", "fortuna_lora.py", "README.md",
           "train.jsonl", "validation.jsonl", "test.jsonl"}
with zipfile.ZipFile(archive_name) as archive:
    if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
        raise ValueError("Неожиданный состав архива.")
    if sum(i.file_size for i in archive.infolist()) > 32 * 1024**2:
        raise ValueError("Слишком большой стартовый набор.")
    for name in sorted(allowed):
        (BUNDLE / name).write_bytes(archive.read(name))
# Проверяем хеш helper до его импорта.
manifest = json.loads((BUNDLE / "manifest.json").read_text(encoding="utf-8"))
if set(manifest["sha256"]) != allowed - {"manifest.json"}:
    raise ValueError("???????? ?????? ????????? ??? ??????? ?????.")
for name, digest in manifest["sha256"].items():
    if name not in allowed or hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() != digest:
        raise ValueError("Нарушена целостность комплекта.")
sys.path.insert(0, str(BUNDLE))
from fortuna_lora import verify_bundle, score_prediction, summarize, render_examples
contract, splits = verify_bundle(BUNDLE)
print("Примеров:", {s: len(r) for s, r in splits.items()})
print("Все задания синтетические. Файловые действия не запускаются.")
shutil.copy2(BUNDLE / "manifest.json", RUN_DIR / "dataset-manifest.json")


## Загрузка исходных весов и подготовка текста

Загружается исходный Hugging Face checkpoint, затем добавляется LoRA. Существующий GGUF приложения не используется для обучения. Текстовая часть модели загружается без vision-компонента; модель этого эксперимента не предназначена для скриншотов.

Примеры проверяются до обучения: переполнение контекста останавливает запуск. Скрытой обрезки правильных ответов нет. Размышление отключено для короткого JSON; общий reasoning-профиль этим набором не обучается.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=contract["model_id"], revision=contract["model_revision"],
    max_seq_length=MAX_SEQ_LENGTH, text_only=True,
    load_in_4bit=False, load_in_16bit=True, full_finetuning=False,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
prepared = {name: render_examples(rows, tokenizer, MAX_SEQ_LENGTH,
    contract["runtime_max_new_tokens"]) for name, rows in splits.items()}
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)
model.print_trainable_parameters()
metadata = {"model_id": contract["model_id"], "revision": contract["model_revision"],
    "smoke_test": SMOKE_TEST, "max_seq_length": MAX_SEQ_LENGTH,
    "gpu": torch.cuda.get_device_name(0), "bf16": USE_BF16,
    "dataset_counts": manifest["counts"], "source_sha256": contract["source_sha256"],
    "packages": {p: version(p) for p in ("unsloth", "unsloth_zoo", "transformers", "trl", "torch", "peft")}}
(RUN_DIR / "run.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "pip-freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")


## Проверка до и после обучения

Метрики проверяют структуру и точное совпадение плана с эталоном. Это не end-to-end выполнение задачи. `plan_match` допускает переименование ID шагов, но равнозначный пересказ аргументов может требовать ручного просмотра. В Colab нет JSON-грамматики llama.cpp, поэтому после GGUF нужен отдельный замер в приложении.


In [ ]:
def evaluate(rows, label):
    FastLanguageModel.for_inference(model)
    results = []
    for row in rows:
        prompt = tokenizer.apply_chat_template(row["messages"], tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=contract["runtime_max_new_tokens"],
                do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        generated = output[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True)
        results.append({"id": row["id"], "category": row["category"],
            **score_prediction(raw, row, contract), "answer": raw,
            "seconds": round(time.perf_counter() - started, 3), "new_tokens": len(generated)})
    (RUN_DIR / f"{label}.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    summary = summarize(results)
    print(label, json.dumps(summary, ensure_ascii=False))
    return summary

validation_rows = splits["validation"][:2] if SMOKE_TEST else splits["validation"]
with model.disable_adapter():
    baseline = evaluate(validation_rows, "validation-base")


## Обучение

Короткий запуск: пять optimizer steps. Полный: одна эпоха. Loss считается только на ответах; пользовательские сообщения и каталог маскируются. Небольшой seed-набор предназначен для проверки процесса. Для выводов о качестве расширьте данные и независимые проверки.


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=Dataset.from_list(prepared["train"]),
    eval_dataset=Dataset.from_list(prepared["validation"]),
    args=SFTConfig(
        output_dir=str(RUN_DIR / "checkpoints"), max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True, packing=False,
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        max_steps=5 if SMOKE_TEST else -1, num_train_epochs=1,
        learning_rate=5e-5, warmup_ratio=0.05, weight_decay=0.01,
        optim="adamw_8bit", bf16=USE_BF16, fp16=not USE_BF16,
        logging_steps=1, save_steps=25, save_total_limit=2,
        eval_strategy="no" if SMOKE_TEST else "epoch", per_device_eval_batch_size=1,
        report_to="none", seed=3407, data_seed=3407,
    ),
)
batch = next(iter(trainer.get_train_dataloader()))
if not (batch["labels"] == -100).any() or not (batch["labels"] != -100).any(dim=1).all():
    raise RuntimeError("Некорректная маска loss. Обучение остановлено.")
del batch
training_result = trainer.train()
trainer.save_model(str(RUN_DIR / "adapter"))
tokenizer.save_pretrained(str(RUN_DIR / "adapter"))
(RUN_DIR / "training-metrics.json").write_text(json.dumps(training_result.metrics, indent=2), encoding="utf-8")
# Сначала сохраняем адаптер; разрыв Colab после этой точки не потеряет итог на диске сеанса.
# До закрытия сеанса скачайте архив последней ячейкой.
del trainer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
adapted = evaluate(validation_rows, "validation-lora")
comparison = {"validation_base": baseline, "validation_lora": adapted,
              "warning": "Planner matching only; no PC actions executed."}
if not SMOKE_TEST:
    # Отложенный test используется после завершения обучения, не для выбора checkpoint.
    with model.disable_adapter():
        comparison["test_base"] = evaluate(splits["test"], "test-base")
    comparison["test_lora"] = evaluate(splits["test"], "test-lora")
else:
    print("Короткий запуск: test пропущен; выводы об улучшении делать рано.")
(RUN_DIR / "comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
print("Изменение plan_match на validation:", round(adapted["plan_match"] - baseline["plan_match"], 3))
print("Просмотрите ответы и категории. Большая точность JSON сама по себе не доказывает улучшение задач.")


## Необязательный экспорт GGUF

Включается флагом `EXPORT_GGUF` после полного запуска. Конвертация требует дополнительной RAM и свободного диска. Этот код не переключает рабочую модель «Фортуны». Узкую LoRA следует проверить также на обычном чате, RAG и проверке источников: в приложении эти вызовы используют тот же chat-server.


In [ ]:
if EXPORT_GGUF:
    if SMOKE_TEST:
        raise RuntimeError("Экспорт короткого пробного запуска выключен. Сначала выполните полный эксперимент.")
    model.save_pretrained_gguf(str(RUN_DIR / "gguf"), tokenizer, quantization_method="q4_k_m")
    print("Новые GGUF:", [str(p) for p in (RUN_DIR / "gguf").rglob("*.gguf")])
else:
    print("GGUF не экспортируется. Адаптер и отчёты готовы к скачиванию.")


## Сохранить результаты

Скачайте архив до завершения сеанса. Он содержит адаптер, tokenizer, настройки и отчёты. Checkpoints и GGUF в этот небольшой архив не включены. Если экспортировали GGUF, сохраните его отдельно через панель файлов Colab. Автоматической публикации на Hugging Face или подключения Google Drive нет.


In [ ]:
download_path = Path("/content/fortuna-adapter-and-reports.zip")
with zipfile.ZipFile(download_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and "checkpoints" not in path.relative_to(RUN_DIR).parts and "gguf" not in path.relative_to(RUN_DIR).parts:
            archive.write(path, path.relative_to(RUN_DIR))
files.download(str(download_path))
